# Analysis 4: Natural Disaster Pattern Analysis

## Objective
To identify meaningful patterns and relationships between natural disaster frequency, severity, affected populations, and economic losses.

## Research Questions
1. Which disaster types are associated with the highest mortality?
2. Which disaster types affect the largest number of people?
3. Which disaster types cause the greatest economic losses?
4. Are frequently occurring disasters necessarily the most destructive?
5. Are there relationships between deaths, affected population, and economic damage?

## Methodology
- Compare disaster types using event-level records.
- Calculate average impacts per reported event.
- Examine relationships between different impact measures.
- Identify disaster types with unusually high average impacts.
- Consider missing data and reporting coverage when interpreting results.

## Limitations
- Missing impact values are not treated as zero.
- Economic damage is reported for fewer events than deaths or affected population.
- Extreme disasters can influence averages significantly.
- Correlation does not establish causation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

ROOT = Path.cwd()

if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "analysis" / "outputs"
VIS_DIR = ROOT / "visualizations" / "patterns"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

events = pd.read_csv(
    DATA_DIR / "natural_disaster_events.csv",
    encoding="utf-8-sig"
)

print("Dataset shape:", events.shape)
print("Disaster types:", events["Disaster Type"].nunique())
events.head()

In [ ]:
impact_columns = [
    "Total Deaths",
    "Total Affected",
    "Total Damage, Adjusted (million US$)"
]

for col in impact_columns:
    events[col] = pd.to_numeric(events[col], errors="coerce")

print("Missing values in impact columns:")
print(events[impact_columns].isnull().sum())

In [ ]:
#DISASTER TYPES ACROSS IMPACTS (IMP)

In [ ]:
pattern_summary = (
    events.groupby("Disaster Type")
    .agg(
        Unique_Events=("Event ID", "nunique"),

        Death_Reported=("Total Deaths", "count"),
        Total_Deaths=("Total Deaths", lambda x: x.sum(min_count=1)),
        Average_Deaths=("Total Deaths", "mean"),
        Median_Deaths=("Total Deaths", "median"),

        Affected_Reported=("Total Affected", "count"),
        Total_Affected=("Total Affected", lambda x: x.sum(min_count=1)),
        Average_Affected=("Total Affected", "mean"),
        Median_Affected=("Total Affected", "median"),

        Damage_Reported=("Total Damage, Adjusted (million US$)", "count"),
        Total_Damage=("Total Damage, Adjusted (million US$)", lambda x: x.sum(min_count=1)),
        Average_Damage=("Total Damage, Adjusted (million US$)", "mean"),
        Median_Damage=("Total Damage, Adjusted (million US$)", "median")
    )
    .reset_index()
)

pattern_summary["Death Coverage (%)"] = (
    pattern_summary["Death_Reported"] / pattern_summary["Unique_Events"] * 100
)

pattern_summary["Affected Coverage (%)"] = (
    pattern_summary["Affected_Reported"] / pattern_summary["Unique_Events"] * 100
)

pattern_summary["Damage Coverage (%)"] = (
    pattern_summary["Damage_Reported"] / pattern_summary["Unique_Events"] * 100
)

pattern_summary.head(15)

In [ ]:
#HIGHEST

In [ ]:
top_death_types = (
    pattern_summary
    .dropna(subset=["Average_Deaths"])
    .sort_values("Average_Deaths", ascending=False)
)

display(
    top_death_types[
        ["Disaster Type", "Unique_Events", "Death_Reported",
         "Average_Deaths", "Median_Deaths", "Death Coverage (%)"]
    ].head(10)
)

top_affected_types = (
    pattern_summary
    .dropna(subset=["Average_Affected"])
    .sort_values("Average_Affected", ascending=False)
)

display(
    top_affected_types[
        ["Disaster Type", "Unique_Events", "Affected_Reported",
         "Average_Affected", "Median_Affected", "Affected Coverage (%)"]
    ].head(10)
)

top_damage_types = (
    pattern_summary
    .dropna(subset=["Average_Damage"])
    .sort_values("Average_Damage", ascending=False)
)

display(
    top_damage_types[
        ["Disaster Type", "Unique_Events", "Damage_Reported",
         "Average_Damage", "Median_Damage", "Damage Coverage (%)"]
    ].head(10)
)


In [ ]:
#AVERAGE

In [ ]:
plt.figure(figsize=(13, 7))

sns.barplot(
    data=top_death_types.head(10),
    x="Average_Deaths",
    y="Disaster Type"
)

plt.title("Top Disaster Types by Average Deaths per Event")
plt.xlabel("Average Deaths per Reported Event")
plt.ylabel("Disaster Type")
plt.tight_layout()

plt.savefig(VIS_DIR / "01_average_deaths_by_type.png", dpi=300)
plt.show()

plt.figure(figsize=(13, 7))

sns.barplot(
    data=top_affected_types.head(10),
    x="Average_Affected",
    y="Disaster Type"
)

plt.title("Top Disaster Types by Average Affected Population")
plt.xlabel("Average Affected People per Reported Event")
plt.ylabel("Disaster Type")
plt.tight_layout()

plt.savefig(VIS_DIR / "02_average_affected_by_type.png", dpi=300)
plt.show()

plt.figure(figsize=(13, 7))

sns.barplot(
    data=top_damage_types.head(10),
    x="Average_Damage",
    y="Disaster Type"
)

plt.title("Top Disaster Types by Average Economic Damage")
plt.xlabel("Average Adjusted Damage (million US$)")
plt.ylabel("Disaster Type")
plt.tight_layout()

plt.savefig(VIS_DIR / "03_average_damage_by_type.png", dpi=300)
plt.show()

In [ ]:
correlation_data = events[
    ["Total Deaths", "Total Affected",
     "Total Damage, Adjusted (million US$)"]
].copy()

correlation_matrix = correlation_data.corr(method="spearman")

print(correlation_matrix)

In [ ]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Between Natural Disaster Impacts")
plt.tight_layout()

plt.savefig(VIS_DIR / "04_impact_correlation_heatmap.png", dpi=300)
plt.show()

In [ ]:
frequency_impact = pattern_summary[
    ["Disaster Type", "Unique_Events", "Average_Deaths",
     "Average_Affected", "Average_Damage"]
].copy()

display(frequency_impact.sort_values("Unique_Events", ascending=False))

In [ ]:
plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=frequency_impact,
    x="Unique_Events",
    y="Average_Deaths",
    hue="Disaster Type",
    s=120
)

plt.title("Disaster Frequency vs Average Deaths")
plt.xlabel("Number of Unique Events")
plt.ylabel("Average Deaths per Reported Event")
plt.tight_layout()

plt.savefig(VIS_DIR / "05_frequency_vs_average_deaths.png", dpi=300)
plt.show()

In [ ]:
pattern_summary.to_csv(
    OUTPUT_DIR / "disaster_pattern_summary.csv",
    index=False
)

frequency_impact.to_csv(
    OUTPUT_DIR / "frequency_vs_destructiveness.csv",
    index=False
)

correlation_matrix.to_csv(
    OUTPUT_DIR / "impact_correlation_matrix.csv"
)

print("Pattern analysis outputs saved successfully!")